# Verify AL5D robot control

Use this notebook to verify the AL5D controller serial connection and the terminal-to-servo wiring. Run the cells in order. **Keep the robot's workspace clear before enabling motion.**

In [ ]:
import os
import sys
from pathlib import Path

project_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'src' / 'exp_run_config.py').is_file()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the BerryPicker repository or one of its subdirectories.')
sys.path.insert(0, str(project_root / 'src'))

from exp_run_config import Config
from robot.al5d_position_controller import PositionController, RobotPosition

Config.PROJECTNAME = 'BerryPicker'
exp_robot_controller = Config().get_experiment('robot_al5d', 'position_controller_00')
exp_pulse_controller = Config().get_experiment(
    exp_robot_controller['exp_pulsecontroller'], exp_robot_controller['run_pulsecontroller']
)
print(f"Robot configuration: robot_al5d/position_controller_00")
print(f"Configured serial device: {exp_pulse_controller['device']}")
print(f"Backup serial device:     {exp_pulse_controller['device_backup']}")

In [ ]:
# This cell does not connect to or move the robot.
for device in (exp_pulse_controller['device'], exp_pulse_controller['device_backup']):
    print(f"{'PASS' if os.path.exists(device) else 'FAIL'}  {device} exists; "
          f"readable={os.access(device, os.R_OK) if os.path.exists(device) else False}; "
          f"writable={os.access(device, os.W_OK) if os.path.exists(device) else False}")

print('If neither device passes, check robot power, USB cabling, and serial permissions before continuing.')

## Enable motion deliberately

Check that power is on, the workspace is clear, and the emergency stop procedure is known. Setting `ALLOW_ROBOT_MOTION` to `True` is required to open `PositionController`; its current constructor homes the robot immediately.

In [ ]:
ALLOW_ROBOT_MOTION = False  # Change to True only after completing the safety checks above.
assert ALLOW_ROBOT_MOTION, 'Set ALLOW_ROBOT_MOTION = True only when it is safe for the robot to move.'

rob = PositionController(exp_robot_controller)
rob.start_robot()
home_position = rob.get_position()
print('PASS  Connected and moved to the configured home position:')
print(home_position)

In [ ]:
# Run one test at a time. Observe the named mechanism, then it returns to home.
# Change test_name only among the values below. Do not increase the offsets without rechecking limits.
test_offsets = {
    'height': -0.5,
    'distance': 0.5,
    'heading': 5.0,
    'wrist_angle': 5.0,
    'wrist_rotation': 5.0,
    'gripper': -10.0,
}
test_name = 'heading'
assert test_name in test_offsets
assert 'rob' in globals(), 'Run the enable-motion cell first.'

target = rob.get_position()
target[test_name] += test_offsets[test_name]
assert RobotPosition.limit(exp_robot_controller, target), f'Unsafe target for {test_name}: {target}'
print(f'Moving {test_name} by {test_offsets[test_name]}. Verify the expected servo moves, then it will return home.')
try:
    rob.move(target)
    input('Observe the motion. Press Enter to return to the recorded home position...')
finally:
    rob.move(home_position)
    print('Returned to home position.')

In [ ]:
# Always run this when verification is complete, or after an interrupted test.
if 'rob' in globals():
    rob.stop_robot()
    del rob
    print('PASS  Robot returned home and servo power was disabled.')
else:
    print('No active robot controller to stop.')